Chuẩn bị

In [ ]:
import os
import json

import pandas as pd

from openai import OpenAI

from pydantic import BaseModel
from pydantic import ValidationError
from typing import List

from day8_data import documents, ground_truth

client = OpenAI()

Định nghĩa schema

In [ ]:
class Item(BaseModel):
    name: str
    quantity: int
    price: float

class Invoice(BaseModel):
    vendor: str
    date: str
    currency: str
    total: float
    items: List[Item]

Viết System Prompt

In [ ]:
SYSTEM_PROMPT = """
You are an information extraction system.

Extract invoice information from the input text.

Return ONLY a valid JSON object.

The JSON must follow exactly this schema:

{
    "vendor": "string",
    "date": "string",
    "currency": "string",
    "total": number,
    "items": [
        {
            "name": "string",
            "quantity": integer,
            "price": number
        }
    ]
}

If a field is missing, use null.

Do not explain anything.
Do not use markdown.
Do not wrap the JSON inside ```json.
"""

Hàm gọi OpenAI API

In [ ]:
def extract_json(text):

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": text
            }
        ]
    )

    return response.choices[0].message.content

Parse JSON an toàn

In [ ]:
def extract(text):

    for attempt in range(2):

        try:

            json_text = extract_json(text)

            data = json.loads(json_text)

            invoice = Invoice(**data)

            return {
                "success": True,
                "need_review": False,
                "data": invoice.model_dump()
            }

        except (json.JSONDecodeError, ValidationError):

            if attempt == 1:

                return {
                    "success": False,
                    "need_review": True,
                    "data": None
                }

Chạy mô hình trên toàn bộ tài liệu

In [ ]:
results = []

for text in documents:
    result = extract(text)
    results.append(result)

Chuyển kết quả thành DataFrame

In [ ]:
rows = []

for result in results:

    if result["success"]:

        rows.append(result["data"])

df = pd.DataFrame(rows)

df

So với ground truth kèm sẵn để đo độ chính xác trích xuất

In [ ]:
correct = 0
total = len(ground_truth)

for result, truth in zip(results, ground_truth):

    if not result["success"]:
        continue

    prediction = result["data"]

    if (
        prediction["vendor"] == truth["vendor"]
        and prediction["date"] == truth["date"]
        and prediction["currency"] == truth["currency"]
        and prediction["total"] == truth["total"]
    ):
        correct += 1

accuracy = correct / total

print(f"Accuracy: {accuracy:.2%}")

Các document cần kiểm tra

In [ ]:
print("Documents needing review:")

for i, result in enumerate(results, start=1):

    if result["need_review"]:
        print(f"- Document {i}")